# Agent Evaluation Demo
In this notebook, we show how to use GAIA validation dataset and web-search tool to evaluate Agent. 
1. First, start PAI-RAG.
2. Then, either call the API or use the frontend UI to add the LLM model and MCP tools, or configure web search.

In [ ]:
from openai import OpenAI

import httpx
import asyncio
import json
from typing import List, Dict, Any

In [ ]:
def get_evaluation_scores(
    answer1: str, answer2: str, model: str = "qwen-max"
) -> int:
    """
    使用 llm model 判断两个答案是否语义相同，返回 0 或 1。

    参数:
        answer1 (str): 第一个答案
        answer2 (str): 第二个答案

    返回:
        int: 1 表示相同，0 表示不同
    """
    openai_api_key = "YOUR_API_KEY"  # 替换为你的 模型 API 密钥
    openai_api_base = "YOUR_API_URL"  # 替换为你的 模型 API URL           


    prompt = f"""请严格判断以下两个答案是否语义完全相同。输出仅包含数字 1（相同）或 0（不同），无需解释。

答案1: {answer1}
答案2: {answer2}"""
    client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "你是一个严格的文本对比工具，专注于判断两个答案是否完全等价。",
                },
                {"role": "user", "content": prompt},
            ],
            max_tokens=1,
            temperature=0,
        )
        result = int(response.choices[0].message.content.strip())
        return result if result in (0, 1) else 0
    except Exception as e:
        print(f"Error: {e}")
        return 0


# 定义请求体模型结构，配置chatbot
def make_chat_request(query: str, model: str = "qwen-max") -> Dict[str, Any]:
    # 1. 推荐用chatbot模式,在配置页面配置chatbot
    # return {
    #     "model": model,
    #     "messages": [{"role": "user", "content": query}],
    #     "stream": True, 
    #     "enable_attachments": False,
    # }
    # 2. 可以直接调用模型
    return {
        "model": model,
        "messages": [{"role": "user", "content": query}],
        "stream": True, 
        "enable_attachments": False,
        "user_id": "", # 这里填写你的USER ID
        "mcp_ids": [],  # 这里填写你使用的MCP ID
        "kb_ids": [],  # 这里填写你使用的知识库 ID 
        "enable_search": True,   # 是否启用网络搜索
        "enable_agent": True,  # 是否启用Agentic模式
        "max_steps": 15, # Agentic模式最大步数
    }


def make_chat_requests(
    queries: List[str], model: str = "qwen-max"
) -> List[Dict[str, Any]]:
    return [make_chat_request(query, model) for query in queries]


def get_queries(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


async def fetch(
    semaphore: asyncio.Semaphore,
    url: str,
    headers: dict,
    data: dict,
    json_line: dict,
    evaluation_model: str = "qwen-max",
) -> dict:
    async with semaphore:  # 控制最大并发数
        async with httpx.AsyncClient(timeout=600) as client:
            try:
                response = await client.post(url, headers=headers, json=data)
                print("Status Code:", response.status_code)

                result_answer = response.json()
                answer_dict = {
                    "task_id": json_line["task_id"],
                    "Level": json_line["Level"],
                    "Agent Answer": result_answer.get("answer", ""),
                    "step": result_answer.get("step", 0),
                    "run_time": f"{result_answer.get('run_time', 0)}s",
                }
                print(
                    "Final Answer:\n",
                    json.dumps(
                        answer_dict["Agent Answer"],
                        indent=2,
                        ensure_ascii=False,
                    ),
                )

                # 调用评分函数
                score = get_evaluation_scores(
                    answer_dict["Agent Answer"],
                    json_line["Final answer"],
                    evaluation_model,
                )
                answer_dict["score"] = score

                return answer_dict
            except Exception as e:
                print(
                    f"Failed to process task_id {json_line['task_id']}: {str(e)}"
                )
                print(
                    "Raw Response:",
                    response.text if "response" in locals() else "No response",
                )
                return {"task_id": json_line["task_id"], "error": str(e)}


async def call_chat_final_answer(
    input_file: str,
    model: str = "qwen-max",
    evaluation_model: str = "qwen-max",
) -> List[Dict[str, Any]]:
    url = "http://localhost:8688/v1/agent/chat_final_answer"  # 替换为你的服务地址
    headers = {"Content-Type": "application/json"}
    # 设置最大并发数（例如 10）
    semaphore = asyncio.Semaphore(20)

    tasks = []
    results = []
    for json_line in get_queries(input_file):
        query = json_line["Question"]
        data = make_chat_request(query, model)
        task = fetch(
            semaphore, url, headers, data, json_line, evaluation_model
        )
        tasks.append(task)

    results = await asyncio.gather(*tasks)
    return results

In [ ]:
import nest_asyncio

nest_asyncio.apply()
input_file = "validation_metadata.jsonl"
output_path = "results_eval.jsonl"
model = "qwen-max"
evaluation_model = "qwen-max"
results = await call_chat_final_answer(input_file, model, evaluation_model)
with open(output_path, "w", encoding="utf-8") as file:
    for line in results:
        file.write(json.dumps(line, ensure_ascii=False) + "\n")